In [24]:
# Clear variables
%reset -f

import duckdb
import pandas as pd

# Constants
DB_PATH = "/home/rud/Documents/01_IE/08_INTR/03_Material_Analysis/project/industrial_cluster.db"
COMPANY_ID = "C005"  # Change this

# Persistent connection with SQLite attached
con = duckdb.connect()
con.execute(f"ATTACH '{DB_PATH}' AS db (TYPE SQLITE)")

In [ ]:
# ── CELL 2: Company header ─────────────────────────────────────────────────────

df_header = con.execute("""
    SELECT
        c.company_id,
        c.name                      AS company_name,
        c.normalize_stream_id       AS reference_stream_id,
        ref.stream_name             AS reference_stream_name,
        ref.flow_kton_per_year      AS reference_flow_kton_per_year
    FROM db.companies c
    LEFT JOIN db.streams ref ON c.normalize_stream_id = ref.stream_id
    WHERE c.company_id = $company_id
""", {"company_id": COMPANY_ID}).df()

display(df_header)

,company_id,company_name,reference_stream_id,reference_stream_name,reference_flow_kton_per_year
0,C005,Biogas Plant,S076,Biogas line 1,769.0


In [26]:
# ── CELL 3: Stream-level mass balance ─────────────────────────────────────────

df_streams = con.execute("""
    SELECT
        s.stream_id,
        s.stream_name,
        s.stream_type,
        s.direction,
        s.flow_kton_per_year,
        s.norm_flow_kton_per_year,
        ROUND(s.carbon_pct * 100, 2)                        AS carbon_pct_display,
        s.carbon_pct_complete,
        ROUND(s.flow_kton_per_year * s.carbon_pct, 3)       AS carbon_flow_kton_per_year,
        s.temperature_c,
        s.pressure_bar,
        s.notes
    FROM db.streams s
    WHERE s.company_id = $company_id
    ORDER BY
        CASE s.direction  WHEN 'input' THEN 0 ELSE 1 END,
        CASE s.stream_type
            WHEN 'raw_material' THEN 0
            WHEN 'product'      THEN 1
            WHEN 'waste'        THEN 2
            ELSE 3
        END,
        s.flow_kton_per_year DESC
""", {"company_id": COMPANY_ID}).df()

df_streams

,stream_id,stream_name,stream_type,direction,flow_kton_per_year,norm_flow_kton_per_year,carbon_pct_display,carbon_pct_complete,carbon_flow_kton_per_year,temperature_c,pressure_bar,notes
0,S075,Dilution water,raw_material,input,13520.0,17.581274,0.00,1,0.000,NaN,NaN,None
1,S074,Food waste,raw_material,input,3300.0,4.291287,NaN,0,NaN,NaN,NaN,None
2,S080,Dilution water,waste,output,12683.0,16.492848,0.00,1,0.000,NaN,NaN,None
3,S079,Liquid effluent,waste,output,2205.0,2.867360,NaN,<NA>,NaN,NaN,NaN,None
4,S078,Cake,waste,output,1095.0,1.423927,NaN,<NA>,NaN,NaN,NaN,None
5,S076,Biogas line 1,products,output,769.0,1.000000,56.65,1,435.665,NaN,NaN,None
6,S077,Biogas line 2,products,output,68.0,0.088427,57.66,1,39.208,NaN,NaN,None


In [27]:
# ── CELL 4: Totals by direction + stream_type (closure check) ─────────────────

df_totals = con.execute("""
    SELECT * FROM (
        SELECT
            direction,
            stream_type,
            COUNT(*)                                            AS stream_count,
            ROUND(SUM(flow_kton_per_year), 3)                  AS total_flow_kton_per_year,
            ROUND(SUM(norm_flow_kton_per_year), 3)             AS total_norm_flow,
            ROUND(SUM(flow_kton_per_year * carbon_pct), 3)     AS total_carbon_flow_kton_per_year
        FROM db.streams
        WHERE company_id = $company_id
          AND flow_kton_per_year IS NOT NULL
        GROUP BY direction, stream_type

        UNION ALL

        SELECT
            direction,
            'TOTAL'                                             AS stream_type,
            COUNT(*)                                            AS stream_count,
            ROUND(SUM(flow_kton_per_year), 3)                  AS total_flow_kton_per_year,
            ROUND(SUM(norm_flow_kton_per_year), 3)             AS total_norm_flow,
            ROUND(SUM(flow_kton_per_year * carbon_pct), 3)     AS total_carbon_flow_kton_per_year
        FROM db.streams
        WHERE company_id = $company_id
          AND flow_kton_per_year IS NOT NULL
        GROUP BY direction
    )
    ORDER BY
        CASE direction   WHEN 'input' THEN 0 ELSE 1 END,
        CASE stream_type WHEN 'TOTAL' THEN 99 ELSE 0 END
""", {"company_id": COMPANY_ID}).df()

df_totals

,direction,stream_type,stream_count,total_flow_kton_per_year,total_norm_flow,total_carbon_flow_kton_per_year
0,input,raw_material,2,16820.0,21.873,0.000
1,input,TOTAL,2,16820.0,21.873,0.000
2,output,waste,3,15983.0,20.784,0.000
3,output,products,2,837.0,1.088,474.873
4,output,TOTAL,5,16820.0,21.873,474.873


In [28]:
# ── CELL 6: Mass balance closure summary ──────────────────────────────────────

totals = (
    df_totals[df_totals["stream_type"] == "TOTAL"]
    .set_index("direction")
)

total_in  = totals.loc["input",  "total_flow_kton_per_year"]
total_out = totals.loc["output", "total_flow_kton_per_year"]
gap       = total_in - total_out
closure   = (1 - abs(gap) / total_in) * 100 if total_in else float("nan")

carbon_in  = totals.loc["input",  "total_carbon_flow_kton_per_year"]
carbon_out = totals.loc["output", "total_carbon_flow_kton_per_year"]
carbon_gap = carbon_in - carbon_out

print(f"Company : {COMPANY_ID}")
print(f"{'─'*40}")
print(f"Total input       : {total_in:>10.3f}  kton/yr")
print(f"Total output      : {total_out:>10.3f}  kton/yr")
print(f"Gap (in − out)    : {gap:>10.3f}  kton/yr")
print(f"Closure           : {closure:>10.2f}  %")
print(f"{'─'*40}")
print(f"Carbon input      : {carbon_in:>10.3f}  kton C/yr")
print(f"Carbon output     : {carbon_out:>10.3f}  kton C/yr")
print(f"Carbon gap        : {carbon_gap:>10.3f}  kton C/yr")

Company : C005
────────────────────────────────────────
Total input       :  16820.000  kton/yr
Total output      :  16820.000  kton/yr
Gap (in − out)    :      0.000  kton/yr
Closure           :     100.00  %
────────────────────────────────────────
Carbon input      :      0.000  kton C/yr
Carbon output     :    474.873  kton C/yr
Carbon gap        :   -474.873  kton C/yr
